In [ ]:

# ===================== Install Dependencies =====================
!pip install transformers seaborn --quiet

# ===================== Imports =====================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.utils import resample
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    confusion_matrix, accuracy_score, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
from torch.utils.tensorboard import SummaryWriter
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from datetime import datetime

# ===================== Load & Balance Dataset =====================
data_path = '/content/SemEval2016-Task6-raw-annotations-stance.csv'
df = pd.read_csv(data_path)
df = df[df['Stance'].isin(['FAVOR', 'AGAINST', 'NONE'])]

label_map = {'FAVOR': 0, 'AGAINST': 1, 'NONE': 2}
df['label'] = df['Stance'].map(label_map)

# Upsample minority classes
df_favor = df[df['label'] == 0]
df_against = df[df['label'] == 1]
df_none = df[df['label'] == 2]
majority_count = len(df_against)

df_favor_up = resample(df_favor, replace=True, n_samples=majority_count, random_state=42)
df_none_up = resample(df_none, replace=True, n_samples=majority_count, random_state=42)

df_balanced = pd.concat([df_against, df_favor_up, df_none_up]).sample(frac=1, random_state=42).reset_index(drop=True)

# ===================== Dataset & DataLoader =====================
class StanceDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        self.labels = labels

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ===================== Tokenizer & Data Preparation =====================
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_balanced['Tweet'].tolist(),
    df_balanced['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_balanced['label']
)
train_dataset = StanceDataset(train_texts, train_labels, tokenizer)
val_dataset = StanceDataset(val_texts, val_labels, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# ===================== Model =====================
class DeBERTa_CNN_BiLSTM_MHA(nn.Module):
    def __init__(self, pretrained_model='microsoft/deberta-v3-base', hidden_size=128, num_classes=3, num_heads=4):
        super(DeBERTa_CNN_BiLSTM_MHA, self).__init__()
        self.deberta = AutoModel.from_pretrained(pretrained_model)
        self.conv1 = nn.Conv1d(768, 128, kernel_size=3, padding=1)
        self.bilstm = nn.LSTM(128, hidden_size, bidirectional=True, batch_first=True)
        self.attn = nn.MultiheadAttention(embed_dim=2 * hidden_size, num_heads=num_heads, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(2 * hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.last_hidden_state                           # (batch, seq_len, 768)
        x = x.permute(0, 2, 1)                                  # (batch, 768, seq_len)
        x = self.conv1(x)                                       # (batch, 128, seq_len)
        x = x.permute(0, 2, 1)                                  # (batch, seq_len, 128)
        x, _ = self.bilstm(x)                                   # (batch, seq_len, 2*hidden)
        x, _ = self.attn(x, x, x)                               # Multihead Attention
        x = self.dropout(x[:, 0, :])                            # CLS-style pooling
        return self.fc(x)

# ===================== Training Setup =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeBERTa_CNN_BiLSTM_MHA().to(device)
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()
writer = SummaryWriter(log_dir=f"runs/stance_{datetime.now().strftime('%Y%m%d_%H%M%S')}")

# ===================== Train & Eval Functions =====================
def train(model, loader):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    report = classification_report(all_labels, all_preds, target_names=label_map.keys(), output_dict=True)
    roc_auc = roc_auc_score(label_binarize(all_labels, classes=[0, 1, 2]), np.array(all_probs), multi_class='ovr')
    return report, roc_auc, all_labels, all_preds, all_probs

# ===================== Training Loop =====================
for epoch in range(10):
    loss = train(model, train_loader)
    report, roc_auc, y_true, y_pred, y_probs = evaluate(model, val_loader)
    writer.add_scalar('Loss/train', loss, epoch)
    writer.add_scalar('AUC/val', roc_auc, epoch)
    print(f"\nEpoch {epoch+1} | Loss: {loss:.4f} | ROC AUC: {roc_auc:.4f}")
    print(classification_report(y_true, y_pred, target_names=label_map.keys()))

# ===================== Final Evaluation =====================
# ROC Curve
fpr, tpr, _ = roc_curve(label_binarize(y_true, classes=[0, 1, 2]).ravel(), np.array(y_probs).ravel())
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC curve (area = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Stance Detection')
plt.legend(loc='lower right')
plt.grid()
plt.show()

# Confusion Matrix
conf_matrix = confusion_matrix(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')
print(f"\nFinal Accuracy: {accuracy:.4f}")
print(f"Final Macro F1 Score: {f1_macro:.4f}")
print("Confusion Matrix:\n", conf_matrix)

plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_map.keys(), yticklabels=label_map.keys())
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Stance Detection')
plt.tight_layout()
plt.show()
